# CT Anatomy Explorer – AI pipeline (Kaggle / Colab)

This notebook runs the **VISTA-3D** model (MONAI) on public CT scans from the **TotalSegmentator** dataset,
compares the AI masks with the expert masks, and prepares small files for the web viewer.

**One-time settings (Kaggle, right-hand panel → Session options):**
1. **Accelerator:** GPU T4 x2 (or GPU P100).
2. **Internet:** On. (Kaggle asks you to verify your phone number first.)

**Day 1 (Thu):** keep `N_CASES = 1` and run the cells one by one (Shift + Enter). Check the picture in Step 6.

**Before leaving on Day 1:** set `N_CASES = 10`, then click **Save Version → Save & Run All (Commit) → Save**.
Kaggle runs everything in the background; you can close the tab.

**Monday:** open the notebook → **Output** (or the saved version) → download `outputs/web_data.zip` and give it to the viewer team.

*Educational project. Not for diagnosis or clinical decisions.*

In [ ]:
# ---- Settings: the only cell you normally change ----
N_CASES = 1          # Day 1 test: 1.   Background run before the break: 10
N_WEB = 10           # scans exported for the website (pick the best 5 on Monday with tools/select_cases.py)
WEB_SPACING_MM = 2.0 # web copies are resampled to this voxel size (bigger = smaller files)

## Step 0 · Install MONAI and check the GPU (about 1 minute)

In [ ]:
!pip install -q --no-deps monai==1.6.1
!pip install -q nibabel pytorch-ignite einops fire
import torch, monai, nibabel
print('MONAI', monai.__version__, '| PyTorch', torch.__version__, '| GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print('WARNING: no GPU. Turn on the GPU accelerator in the notebook settings, then restart the session.')

## Pipeline code
These cells only write the pipeline files to disk. You do not need to edit them (the same code lives in the `pipeline/` folder of the GitHub repo).

In [ ]:
!mkdir -p pipeline

In [ ]:
%%writefile pipeline/__init__.py
"""CT Anatomy Explorer pipeline: TotalSegmentator data -> VISTA-3D -> metrics -> web viewer files."""


In [ ]:
%%writefile pipeline/organs.py
"""The 12 organs shown in CT Anatomy Explorer v0.5.

Every other file uses this one table, so the mapping between
  * the compact index used in our label maps (1..12),
  * the TotalSegmentator v2 file name (segmentations/<key>.nii.gz), and
  * the VISTA-3D label id (MONAI bundle `vista3d`, configs/metadata.json)
lives in exactly one place.

VISTA-3D ids were copied from the bundle's metadata.json (channel_def).
TotalSegmentator names were checked against totalsegmentator/map_to_binary.py.
If you add organs, add them here and nowhere else.
"""
from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Organ:
    index: int          # value in OUR compact label maps (reference.nii.gz / ai.nii.gz)
    key: str            # TotalSegmentator v2 file name (without .nii.gz)
    name: str           # display name
    vista_id: int       # VISTA-3D label id (prompt + output value)
    color: tuple        # RGB 0-255 used by the viewer and QC figures
    tubular: bool = False  # long structures that may legitimately run out of the scan


ORGANS: list[Organ] = [
    Organ(1, "liver", "Liver", 1, (196, 98, 72)),
    Organ(2, "spleen", "Spleen", 3, (148, 103, 189)),
    Organ(3, "pancreas", "Pancreas", 4, (245, 197, 66)),
    Organ(4, "kidney_right", "Right kidney", 5, (214, 90, 140)),
    Organ(5, "kidney_left", "Left kidney", 14, (236, 139, 178)),
    Organ(6, "gallbladder", "Gallbladder", 10, (85, 168, 104)),
    Organ(7, "stomach", "Stomach", 12, (230, 150, 90)),
    Organ(8, "aorta", "Aorta", 6, (220, 40, 40), tubular=True),
    Organ(9, "inferior_vena_cava", "Inferior vena cava", 7, (60, 110, 220), tubular=True),
    Organ(10, "esophagus", "Oesophagus", 11, (199, 132, 104), tubular=True),
    Organ(11, "duodenum", "Duodenum", 13, (255, 214, 150)),
    Organ(12, "urinary_bladder", "Urinary bladder", 15, (80, 200, 220)),
]

N_ORGANS = len(ORGANS)
VISTA_PROMPTS = [o.vista_id for o in ORGANS]          # label_prompt passed to VISTA-3D
VISTA_TO_INDEX = {o.vista_id: o.index for o in ORGANS}  # VISTA-3D output value -> our index
BY_KEY = {o.key: o for o in ORGANS}
BY_INDEX = {o.index: o for o in ORGANS}

# Colours for the comparison view (Okabe-Ito palette, colour-blind safe)
COMPARE = {
    1: ("Agree (AI and expert)", (0, 158, 115)),
    2: ("AI only (extra)", (213, 94, 0)),
    3: ("Missed by AI", (0, 114, 178)),
}


def organ_table() -> list[dict]:
    """JSON-friendly organ list for the viewer manifest."""
    return [
        {"index": o.index, "key": o.key, "name": o.name, "vista_id": o.vista_id, "color": list(o.color)}
        for o in ORGANS
    ]


In [ ]:
%%writefile pipeline/dataset.py
"""Download the TotalSegmentator small subset and choose good demo cases.

Dataset: "Small subset of TotalSegmentator Dataset" v2.0.1 (102 CT scans,
expert-checked masks for 117 structures), Zenodo record 10047263, CC BY 4.0.
Please credit: Wasserthal J. et al., TotalSegmentator, Radiology: AI 2023.

Expected layout after extraction (searched recursively, so an extra top folder is fine):
    <data_dir>/.../s0011/ct.nii.gz
    <data_dir>/.../s0011/segmentations/liver.nii.gz  (one file per structure)
    <data_dir>/.../meta.csv                           (optional, ';' or ',' separated)
"""
from __future__ import annotations

import csv
import hashlib
import json
import math
import os
import time
import zipfile
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import nibabel as nib
import numpy as np

from .organs import ORGANS

ZIP_NAME = "Totalsegmentator_dataset_small_v201.zip"
ZIP_URL = f"https://zenodo.org/records/10047263/files/{ZIP_NAME}?download=1"
ZIP_MD5 = "6b5524af4b15e6ba06ef2d700c0c73e0"

ORGAN_FILES = {f"segmentations/{o.key}.nii.gz" for o in ORGANS}


# --------------------------------------------------------------------------- download
def _download(url: str, dest: Path, chunk: int = 1 << 20) -> None:
    """Stream a large file with resume support and a simple progress print."""
    import requests  # available on Kaggle/Colab

    dest.parent.mkdir(parents=True, exist_ok=True)
    part = dest.with_suffix(dest.suffix + ".part")
    have = part.stat().st_size if part.exists() else 0
    headers = {"Range": f"bytes={have}-"} if have else {}
    with requests.get(url, stream=True, headers=headers, timeout=60) as r:
        if r.status_code == 200 and have:
            have = 0  # server ignored the range request: start again
        r.raise_for_status()
        total = int(r.headers.get("Content-Length", 0)) + have
        mode = "ab" if have else "wb"
        t0, last = time.time(), -1
        with open(part, mode) as f:
            for block in r.iter_content(chunk_size=chunk):
                f.write(block)
                have += len(block)
                if total:
                    pct = int(100 * have / total)
                    if pct // 5 != last // 5:
                        last = pct
                        speed = have / max(time.time() - t0, 1e-6) / 1e6
                        print(f"  downloaded {have/1e9:5.2f} / {total/1e9:.2f} GB ({pct}%)  ~{speed:.0f} MB/s", flush=True)
    part.rename(dest)


def _md5(path: Path, chunk: int = 1 << 24) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def prepare(data_dir: str | Path, zip_path: str | Path | None = None, verify_md5: bool = True) -> Path:
    """Make sure the dataset is extracted under data_dir. Returns data_dir.

    Only ct.nii.gz, the 12 organ masks and meta.csv are extracted (much faster
    than unpacking all 117 masks per scan).
    """
    data_dir = Path(data_dir)
    marker = data_dir / ".extracted"
    if marker.exists():
        print(f"Dataset already extracted in {data_dir}")
        return data_dir
    data_dir.mkdir(parents=True, exist_ok=True)

    zp = Path(zip_path) if zip_path else data_dir / ZIP_NAME
    if not zp.exists():
        print(f"Downloading {ZIP_NAME} (3.2 GB) from Zenodo ...")
        _download(ZIP_URL, zp)
    if verify_md5:
        print("Checking file integrity (MD5) ...")
        got = _md5(zp)
        if got != ZIP_MD5:
            raise RuntimeError(f"MD5 mismatch for {zp} (got {got}). Delete the file and download again.")

    print("Extracting CT scans, the 12 organ masks and meta.csv ...")
    n = 0
    with zipfile.ZipFile(zp) as z:
        for m in z.infolist():
            name = m.filename.replace("\\", "/")
            if name.endswith("/ct.nii.gz") or name.endswith("meta.csv") or any(name.endswith(s) for s in ORGAN_FILES):
                z.extract(m, data_dir)
                n += 1
    marker.write_text(json.dumps({"files": n, "zip": str(zp)}))
    print(f"Extracted {n} files into {data_dir}")
    return data_dir


# --------------------------------------------------------------------------- discovery
def find_cases(data_dir: str | Path) -> dict[str, Path]:
    """Map case id (folder name, e.g. 's0011') -> case folder."""
    cases = {}
    for ct in sorted(Path(data_dir).rglob("ct.nii.gz")):
        if (ct.parent / "segmentations").is_dir():
            cases[ct.parent.name] = ct.parent
    return cases


def read_meta(data_dir: str | Path) -> dict[str, dict]:
    """Read meta.csv (age, gender, study type ...) if present."""
    files = sorted(Path(data_dir).rglob("meta.csv"))
    if not files:
        return {}
    text = files[0].read_text(encoding="utf-8", errors="replace")
    delim = ";" if text.count(";") > text.count(",") else ","
    rows = csv.DictReader(text.splitlines(), delimiter=delim)
    out = {}
    for r in rows:
        r = {(k or "").strip(): (v or "").strip() for k, v in r.items()}
        if r.get("image_id"):
            out[r["image_id"]] = r
    return out


def describe(meta: dict | None) -> str:
    """Short human description, e.g. 'Female, 63 y · ct abdomen'."""
    if not meta:
        return ""
    sex = {"m": "Male", "f": "Female"}.get(meta.get("gender", "").lower(), "")
    age = meta.get("age", "")
    age = f"{int(float(age))} y" if age.replace(".", "", 1).isdigit() else ""
    study = meta.get("study_type", "")
    first = ", ".join(x for x in (sex, age) if x)
    return " · ".join(x for x in (first, study) if x)


# --------------------------------------------------------------------------- curation
def _si_axis(affine) -> int:
    codes = nib.aff2axcodes(affine)
    return next(i for i, c in enumerate(codes) if c in ("S", "I"))


def assess_case(case_id: str, case_dir: str, min_ml: float = 1.0) -> dict:
    """Check which of the 12 organs are fully visible in one scan."""
    case_dir = Path(case_dir)
    ct = nib.load(str(case_dir / "ct.nii.gz"))
    zooms = [float(z) for z in ct.header.get_zooms()[:3]]
    si = _si_axis(ct.affine)
    row = {
        "case_id": case_id,
        "shape": "x".join(str(s) for s in ct.shape[:3]),
        "spacing_mm": "x".join(f"{z:.2f}" for z in zooms),
        "slice_mm": round(zooms[si], 2),
        "length_mm": round(ct.shape[si] * zooms[si]),
        "voxels_M": round(math.prod(ct.shape[:3]) / 1e6, 1),
    }
    voxel_ml = math.prod(zooms) / 1000.0
    n_ok = 0
    for o in ORGANS:
        p = case_dir / "segmentations" / f"{o.key}.nii.gz"
        if not p.exists():
            status = "no_file"
        else:
            m = np.asarray(nib.load(str(p)).dataobj) > 0
            n = int(m.sum())
            if n == 0:
                status = "absent"
            elif n * voxel_ml < min_ml:
                status = "tiny"
            else:
                other = tuple(a for a in range(3) if a != si)
                idx = np.flatnonzero(m.any(axis=other))
                cut = idx[0] == 0 or idx[-1] == m.shape[si] - 1
                status = "cut_off" if (cut and not o.tubular) else "ok"
        row[o.key] = status
        n_ok += status == "ok"
    row["organs_ok"] = n_ok
    return row


def curate(data_dir: str | Path, work_dir: str | Path, n_cases: int = 10, min_length_mm: int = 300,
           max_voxels_M: float = 150.0, workers: int | None = None) -> list[str]:
    """Rank all scans and pick the best n_cases. Writes curation.csv and selected_cases.json."""
    work_dir = Path(work_dir)
    work_dir.mkdir(parents=True, exist_ok=True)
    cases = find_cases(data_dir)
    meta = read_meta(data_dir)
    if not cases:
        raise RuntimeError(f"No cases found under {data_dir}. Did the download/extraction finish?")

    # quick pre-filter on the CT header only (short scans cannot contain all 12 organs)
    candidates, skipped = [], []
    for cid, cdir in cases.items():
        ct = nib.load(str(cdir / "ct.nii.gz"))
        zooms = ct.header.get_zooms()[:3]
        si = _si_axis(ct.affine)
        length = ct.shape[si] * float(zooms[si])
        vox = math.prod(ct.shape[:3]) / 1e6
        if length < min_length_mm or vox > max_voxels_M:
            skipped.append(cid)
        else:
            candidates.append(cid)
    print(f"{len(cases)} scans found; checking organ masks in {len(candidates)} long-enough scans ...")

    workers = workers or max(1, min(4, os.cpu_count() or 1))
    with ProcessPoolExecutor(max_workers=workers) as ex:
        rows = list(ex.map(assess_case, candidates, [str(cases[c]) for c in candidates]))

    for r in rows:
        m = meta.get(r["case_id"], {})
        r["description"] = describe(m)
        r["pathology"] = m.get("pathology", "")
    rows.sort(key=lambda r: (-r["organs_ok"], r["slice_mm"], r["voxels_M"], r["case_id"]))
    perfect = [r for r in rows if r["organs_ok"] == len(ORGANS)]
    chosen = (perfect + [r for r in rows if r not in perfect])[:n_cases]
    chosen_ids = [r["case_id"] for r in chosen]
    if len(perfect) < n_cases:
        print(f"Note: only {len(perfect)} scans show all {len(ORGANS)} organs fully; filling with the next best.")

    keys = ["selected", "case_id", "organs_ok", "description", "pathology", "shape", "spacing_mm",
            "slice_mm", "length_mm", "voxels_M"] + [o.key for o in ORGANS]
    with open(work_dir / "curation.csv", "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        for r in rows:
            w.writerow({**r, "selected": r["case_id"] in chosen_ids})
    (work_dir / "selected_cases.json").write_text(json.dumps(chosen_ids, indent=1))
    print(f"Selected {len(chosen_ids)} cases: {', '.join(chosen_ids)}  (details: {work_dir/'curation.csv'})")
    return chosen_ids


In [ ]:
%%writefile pipeline/run_vista3d.py
"""Run the MONAI VISTA-3D bundle on selected CT scans (12 organ prompts).

Each scan runs in its own `python -m monai.bundle run` process, so one failure
(or an out-of-memory error) does not stop the whole batch, and a re-run skips
scans that are already done. Results go to <work_dir>/ai_raw/<case_id>.nii.gz
with VISTA-3D label ids as voxel values (e.g. liver = 1, spleen = 3).
"""
from __future__ import annotations

import csv
import os
import shutil
import subprocess
import sys
import threading
import time
from pathlib import Path

from .organs import VISTA_PROMPTS

BUNDLE_REPO = "MONAI/vista3d"          # Hugging Face repo of the MONAI bundle (includes weights)
BUNDLE_REVISION = "0.5.11"             # tested version; falls back to the latest if missing
WEIGHTS_URL = "https://developer.download.nvidia.com/assets/Clara/monai/tutorials/model_zoo/model_vista3d.pt"


def ensure_bundle(bundle_dir: str | Path) -> Path:
    """Download the VISTA-3D bundle (configs, scripts and ~870 MB weights) once."""
    bundle_dir = Path(bundle_dir)
    cfg = bundle_dir / "configs" / "inference.json"
    weights = bundle_dir / "models" / "model.pt"
    if cfg.exists() and weights.exists() and weights.stat().st_size > 500e6:
        print(f"VISTA-3D bundle ready in {bundle_dir}")
        return bundle_dir
    bundle_dir.mkdir(parents=True, exist_ok=True)
    try:
        from huggingface_hub import snapshot_download
        print(f"Downloading VISTA-3D bundle from Hugging Face ({BUNDLE_REPO}) ...")
        try:
            snapshot_download(repo_id=BUNDLE_REPO, revision=BUNDLE_REVISION, local_dir=str(bundle_dir))
        except Exception:
            print(f"  version {BUNDLE_REVISION} not found, using the latest version")
            snapshot_download(repo_id=BUNDLE_REPO, local_dir=str(bundle_dir))
    except Exception as e:  # noqa: BLE001
        print(f"Hugging Face download failed: {e}")
    if cfg.exists() and not (weights.exists() and weights.stat().st_size > 500e6):
        print("Weights missing; downloading model.pt from NVIDIA ...")
        import requests
        weights.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(WEIGHTS_URL, stream=True, timeout=60) as r:
            r.raise_for_status()
            with open(weights, "wb") as f:
                for block in r.iter_content(1 << 20):
                    f.write(block)
    if not (cfg.exists() and weights.exists()):
        raise RuntimeError("Could not get the VISTA-3D bundle. Check that Internet is ON in the notebook settings.")
    print(f"VISTA-3D bundle ready in {bundle_dir}")
    return bundle_dir


def _gpu_used_mb() -> int | None:
    try:
        out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return int(out.stdout.strip().splitlines()[0])
    except Exception:  # noqa: BLE001
        return None


def run_case(case_id: str, ct_path: str | Path, bundle_dir: str | Path, work_dir: str | Path,
             prompts: list[int] | None = None, timeout_s: int = 3600,
             resample_spacing: list[float] | None = None) -> dict:
    """Segment one scan. Returns a record with status, time and peak GPU memory."""
    bundle_dir, work_dir = Path(bundle_dir).resolve(), Path(work_dir).resolve()
    out_path = work_dir / "ai_raw" / f"{case_id}.nii.gz"
    if out_path.exists():
        return {"case_id": case_id, "status": "done_before", "seconds": "", "peak_gpu_mb": "", "output": str(out_path)}
    prompts = prompts or VISTA_PROMPTS

    # The bundle names outputs after the input file, and every TotalSegmentator scan is
    # called ct.nii.gz - so link each scan under a unique name first.
    inputs = work_dir / "_inputs"
    inputs.mkdir(parents=True, exist_ok=True)
    link = inputs / f"{case_id}.nii.gz"
    if not link.exists():
        try:
            link.symlink_to(Path(ct_path).resolve())
        except OSError:
            shutil.copy(ct_path, link)
    raw_dir = work_dir / "_bundle_out"
    logs = work_dir / "logs"
    logs.mkdir(parents=True, exist_ok=True)
    log_path = logs / f"{case_id}.log"

    cmd = [sys.executable, "-m", "monai.bundle", "run",
           "--config_file", str(bundle_dir / "configs" / "inference.json"),
           "--meta_file", str(bundle_dir / "configs" / "metadata.json"),
           "--logging_file", str(bundle_dir / "configs" / "logging.conf"),
           "--bundle_root", str(bundle_dir),
           "--output_dir", str(raw_dir),
           "--output_dtype", "$np.uint8",
           "--input_dict", repr({"image": str(link), "label_prompt": [int(p) for p in prompts]})]
    if resample_spacing:  # only for quick tests; VISTA-3D is meant to run at 1.5 mm
        cmd += ["--resample_spacing", repr([float(s) for s in resample_spacing])]
    env = dict(os.environ, PYTHONPATH=str(bundle_dir) + os.pathsep + os.environ.get("PYTHONPATH", ""))

    base = _gpu_used_mb()
    peak = {"mb": base}
    stop = threading.Event()

    def sample():
        while not stop.is_set():
            v = _gpu_used_mb()
            if v is not None and (peak["mb"] is None or v > peak["mb"]):
                peak["mb"] = v
            stop.wait(2)

    t0 = time.time()
    sampler = threading.Thread(target=sample, daemon=True)
    sampler.start()
    with open(log_path, "w") as log:
        try:
            proc = subprocess.run(cmd, cwd=str(bundle_dir), env=env, stdout=log, stderr=subprocess.STDOUT,
                                  timeout=timeout_s)
            status = "ok" if proc.returncode == 0 else f"failed (exit {proc.returncode})"
        except subprocess.TimeoutExpired:
            status = "timeout"
    stop.set()
    sampler.join(timeout=5)
    seconds = round(time.time() - t0, 1)

    produced = sorted(raw_dir.glob(f"{case_id}/*_trans.nii.gz")) or sorted(raw_dir.glob(f"**/{case_id}*_trans.nii.gz"))
    if status == "ok" and produced:
        out_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(produced[0]), out_path)
        shutil.rmtree(produced[0].parent, ignore_errors=True)
    elif status == "ok":
        status = "failed (no output file)"
    peak_mb = (peak["mb"] - base) if (peak["mb"] is not None and base is not None) else ""
    rec = {"case_id": case_id, "status": status, "seconds": seconds, "peak_gpu_mb": peak_mb,
           "output": str(out_path) if out_path.exists() else "", "log": str(log_path)}
    if not status.startswith("ok"):
        tail = log_path.read_text(errors="replace").strip().splitlines()[-15:]
        rec["error"] = " | ".join(tail)[-1500:]
    return rec


def run_cases(case_ids: list[str], cases: dict[str, Path], bundle_dir: str | Path, work_dir: str | Path,
              **kw) -> list[dict]:
    """Segment several scans one after another and write inference_log.csv."""
    work_dir = Path(work_dir)
    records = []
    for i, cid in enumerate(case_ids, 1):
        print(f"[{i}/{len(case_ids)}] VISTA-3D on {cid} ...", flush=True)
        rec = run_case(cid, cases[cid] / "ct.nii.gz", bundle_dir, work_dir, **kw)
        records.append(rec)
        extra = f"  {rec['seconds']} s" if rec["seconds"] != "" else ""
        extra += f", peak GPU {rec['peak_gpu_mb']} MB" if rec.get("peak_gpu_mb") not in ("", None) else ""
        print(f"    -> {rec['status']}{extra}", flush=True)
        if rec.get("error"):
            print("    last log lines:", rec["error"][-600:])
    keys = ["case_id", "status", "seconds", "peak_gpu_mb", "output", "log", "error"]
    log_csv = work_dir / "inference_log.csv"
    old = {}
    if log_csv.exists():
        with open(log_csv) as f:
            old = {r["case_id"]: r for r in csv.DictReader(f)}
    for r in records:
        if r["status"] != "done_before" or r["case_id"] not in old:
            old[r["case_id"]] = r
    with open(log_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys, extrasaction="ignore")
        w.writeheader()
        w.writerows(old.values())
    ok = sum(1 for r in records if r["status"] in ("ok", "done_before"))
    print(f"VISTA-3D finished: {ok}/{len(records)} scans segmented. Log: {log_csv}")
    return records


In [ ]:
%%writefile pipeline/evaluate.py
"""Compare VISTA-3D masks with the expert reference masks.

For every organ: Dice overlap, volumes (mL) and voxel counts of agreement,
AI-only (extra) and missed. Also runs two automatic sanity checks that catch the
most common silent bugs: a label-number mix-up and a left/right kidney swap.
Scores are *agreement with reference labels*, not proof of clinical accuracy
(VISTA-3D was trained on large public CT collections that may overlap with this data).
"""
from __future__ import annotations

import csv
import json
import math
from pathlib import Path

import nibabel as nib
import numpy as np
from nibabel.processing import resample_from_to

from .organs import BY_KEY, BY_INDEX, N_ORGANS, ORGANS, VISTA_TO_INDEX


def reference_labels(case_dir: str | Path) -> tuple[np.ndarray, nib.Nifti1Image]:
    """Combine the 12 expert masks into one label map (values 1..12) on the CT grid."""
    case_dir = Path(case_dir)
    ct = nib.load(str(case_dir / "ct.nii.gz"))
    lab = np.zeros(ct.shape[:3], np.uint8)
    for o in ORGANS:
        p = case_dir / "segmentations" / f"{o.key}.nii.gz"
        if p.exists():
            lab[np.asarray(nib.load(str(p)).dataobj) > 0] = o.index
    return lab, ct


def ai_labels(ai_path: str | Path, ct: nib.Nifti1Image) -> tuple[np.ndarray, list[str]]:
    """Load a VISTA-3D output and convert its label ids to our 1..12 values on the CT grid."""
    notes = []
    img = nib.load(str(ai_path))
    data = np.asarray(img.dataobj)
    if data.ndim == 4:
        data = data[..., 0]
    if data.shape != ct.shape[:3] or not np.allclose(img.affine, ct.affine, atol=1e-2):
        notes.append("AI mask grid differed from the CT grid and was resampled (nearest neighbour)")
        img = resample_from_to(nib.Nifti1Image(data.astype(np.float32), img.affine), (ct.shape[:3], ct.affine), order=0)
        data = np.asarray(img.dataobj)
    data = np.clip(np.rint(data), 0, 255).astype(np.uint8)
    lut = np.zeros(256, np.uint8)
    for vid, idx in VISTA_TO_INDEX.items():
        lut[vid] = idx
    unexpected = sorted(set(np.unique(data).tolist()) - set(VISTA_TO_INDEX) - {0, 255})
    if unexpected:
        notes.append(f"unexpected label values in AI output: {unexpected[:10]}")
    return lut[data], notes


def confusion(ref: np.ndarray, ai: np.ndarray) -> np.ndarray:
    """(13 x 13) voxel counts: rows = reference label, columns = AI label (0 = background)."""
    k = N_ORGANS + 1
    combo = ref.astype(np.int64).ravel() * k + ai.astype(np.int64).ravel()
    return np.bincount(combo, minlength=k * k).reshape(k, k)


def organ_metrics(cm: np.ndarray, voxel_ml: float) -> dict:
    out = {}
    for o in ORGANS:
        i = o.index
        tp = int(cm[i, i])
        n_ref, n_ai = int(cm[i, :].sum()), int(cm[:, i].sum())
        dice = (2 * tp / (n_ref + n_ai)) if (n_ref + n_ai) else None
        v_ref, v_ai = n_ref * voxel_ml, n_ai * voxel_ml
        out[o.key] = {
            "name": o.name,
            "dice": None if dice is None else round(dice, 4),
            "vol_ref_ml": round(v_ref, 1),
            "vol_ai_ml": round(v_ai, 1),
            "vol_diff_pct": round(100 * (v_ai - v_ref) / v_ref, 1) if v_ref > 0 else None,
            "agree_vox": tp, "ai_only_vox": n_ai - tp, "missed_vox": n_ref - tp,
        }
    return out


def sanity_flags(cm: np.ndarray, metrics: dict) -> list[str]:
    """Warnings that usually mean a bug rather than a weak model."""
    flags = []
    for o in ORGANS:
        m = metrics[o.key]
        if m["vol_ref_ml"] > 0 and m["vol_ai_ml"] > 0 and (m["dice"] or 0) < 0.3:
            col = cm[1:, o.index]
            j = int(np.argmax(col)) + 1
            if j != o.index and col[j - 1] > 0:
                flags.append(f"LABEL MIX-UP? AI '{o.name}' mostly overlaps expert '{BY_INDEX[j].name}'")
            else:
                flags.append(f"Low agreement for {o.name} (Dice {m['dice']}) - check alignment")
        if m["vol_ref_ml"] > 0 and m["vol_ai_ml"] == 0:
            flags.append(f"AI found no {o.name} although the expert marked it")
    r, l = BY_KEY["kidney_right"].index, BY_KEY["kidney_left"].index
    same, cross = cm[r, r] + cm[l, l], cm[r, l] + cm[l, r]
    if cross > same and cross > 0:
        flags.append("LEFT/RIGHT SWAP? Kidneys overlap the opposite side - check orientation")
    return flags


def evaluate_case(case_id: str, case_dir: str | Path, ai_path: str | Path, work_dir: str | Path) -> dict:
    ref, ct = reference_labels(case_dir)
    ai, notes = ai_labels(ai_path, ct)
    zooms = [float(z) for z in ct.header.get_zooms()[:3]]
    voxel_ml = math.prod(zooms) / 1000.0
    cm = confusion(ref, ai)
    metrics = organ_metrics(cm, voxel_ml)
    flags = sanity_flags(cm, metrics)
    dices = [m["dice"] for m in metrics.values() if m["dice"] is not None and m["vol_ref_ml"] > 0]
    result = {
        "case_id": case_id,
        "mean_dice": round(float(np.mean(dices)), 4) if dices else None,
        "organs": metrics,
        "flags": flags,
        "notes": notes,
        "voxel_ml": voxel_ml,
        "spacing_mm": zooms,
        "shape": list(ct.shape[:3]),
    }
    out = Path(work_dir) / "metrics"
    out.mkdir(parents=True, exist_ok=True)
    (out / f"{case_id}.json").write_text(json.dumps(result, indent=1))
    return result


def evaluate_cases(case_ids: list[str], cases: dict[str, Path], work_dir: str | Path) -> list[dict]:
    work_dir = Path(work_dir)
    results, rows = [], []
    for cid in case_ids:
        ai_path = work_dir / "ai_raw" / f"{cid}.nii.gz"
        if not ai_path.exists():
            print(f"  {cid}: no AI output yet - skipped")
            continue
        r = evaluate_case(cid, cases[cid], ai_path, work_dir)
        results.append(r)
        print(f"  {cid}: mean Dice {r['mean_dice']}" + ("".join(f"\n     ! {f}" for f in r["flags"])))
        for key, m in r["organs"].items():
            rows.append({"case_id": cid, "organ": m["name"], "dice": m["dice"], "expert_ml": m["vol_ref_ml"],
                         "ai_ml": m["vol_ai_ml"], "diff_pct": m["vol_diff_pct"], "agree_vox": m["agree_vox"],
                         "ai_only_vox": m["ai_only_vox"], "missed_vox": m["missed_vox"]})
    if rows:
        with open(work_dir / "metrics_all.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
            w.writeheader()
            w.writerows(rows)
        print(f"Metrics for {len(results)} scans written to {work_dir/'metrics_all.csv'}")
    return results


def qc_figure(case_id: str, case_dir: str | Path, ai_path: str | Path | None, out_png: str | Path) -> Path:
    """Side-by-side picture (CT | expert | AI) for a quick visual check. Radiological view:
    patient's right on the image's left, anterior at the top."""
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    from matplotlib.colors import ListedColormap

    ref, ct = reference_labels(case_dir)
    ai = ai_labels(ai_path, ct)[0] if ai_path and Path(ai_path).exists() else None
    ct_c = nib.as_closest_canonical(ct)
    to_can = lambda a: np.asarray(nib.as_closest_canonical(nib.Nifti1Image(a, ct.affine)).dataobj)  # noqa: E731
    img = np.asarray(ct_c.dataobj, dtype=np.float32)
    ref_c = to_can(ref)
    ai_c = to_can(ai) if ai is not None else None
    liver = ref_c == BY_KEY["liver"].index
    z = int(np.argmax(liver.sum(axis=(0, 1)))) if liver.any() else img.shape[2] // 2
    y = int(np.argmax((ref_c > 0).sum(axis=(0, 2)))) if ref_c.any() else img.shape[1] // 2

    colors = np.zeros((N_ORGANS + 1, 4))
    for o in ORGANS:
        colors[o.index] = [*(np.array(o.color) / 255), 0.55]
    cmap = ListedColormap(colors)
    ax_sl = lambda a: a[:, :, z].T[:, ::-1]  # noqa: E731  (rows: P->A with origin lower, cols flipped: R on left)
    co_sl = lambda a: a[:, y, :].T[:, ::-1]  # noqa: E731
    panels = [("CT", None), ("Expert (reference)", ref_c), ("AI (VISTA-3D)", ai_c)]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8.5), facecolor="black")
    for col, (title, lab) in enumerate(panels):
        for row, sl in enumerate((ax_sl, co_sl)):
            ax = axes[row, col]
            ax.imshow(np.clip(sl(img), -160, 240), cmap="gray", origin="lower", interpolation="bilinear",
                      aspect=ct_c.header.get_zooms()[1] / ct_c.header.get_zooms()[0] if row == 0
                      else ct_c.header.get_zooms()[2] / ct_c.header.get_zooms()[0])
            if lab is not None:
                ax.imshow(sl(lab), cmap=cmap, vmin=0, vmax=N_ORGANS, origin="lower", interpolation="nearest",
                          aspect=ax.get_aspect())
            elif title.startswith("AI"):
                ax.text(0.5, 0.5, "no AI output yet", color="w", ha="center", transform=ax.transAxes)
            ax.set_axis_off()
            if row == 0:
                ax.set_title(title, color="w")
    fig.suptitle(f"{case_id} - axial and coronal views (R on left)", color="w")
    fig.tight_layout()
    out_png = Path(out_png)
    out_png.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_png, dpi=90, facecolor="black")
    plt.close(fig)
    return out_png


In [ ]:
%%writefile pipeline/export_web.py
"""Make small, web-ready copies of each case for the NiiVue viewer.

For every case it writes into <web_dir>/<case_id>/:
    ct.nii.gz          CT, cropped to the body, resampled (default 2 mm), int16
    reference.nii.gz   expert masks, values 1..12, same grid as ct.nii.gz
    ai.nii.gz          VISTA-3D masks, values 1..12, same grid (if available)
    ai_fullres.nii.gz  VISTA-3D masks on the ORIGINAL CT grid (for download)
    metrics.csv        per-organ Dice and volumes
and a manifest <web_dir>/cases.json that the viewer reads.
"""
from __future__ import annotations

import csv
import datetime as dt
import json
import shutil
from pathlib import Path

import nibabel as nib
import numpy as np
from nibabel.processing import resample_from_to, resample_to_output
from scipy import ndimage as ndi

from .organs import COMPARE, ORGANS, organ_table
from .evaluate import ai_labels, reference_labels

CT_WINDOW = (-160, 240)  # abdomen window used as the default display


def _bbox(mask: np.ndarray, min_count: int = 20, margin: int = 4) -> list[slice]:
    sl = []
    for ax in range(3):
        other = tuple(a for a in range(3) if a != ax)
        counts = mask.sum(axis=other)
        idx = np.flatnonzero(counts >= min_count)
        if idx.size == 0:
            idx = np.array([0, mask.shape[ax] - 1])
        sl.append(slice(max(int(idx[0]) - margin, 0), min(int(idx[-1]) + margin + 1, mask.shape[ax])))
    return sl


def _union(a: list[slice], b: list[slice]) -> list[slice]:
    return [slice(min(x.start, y.start), max(x.stop, y.stop)) for x, y in zip(a, b)]


def _interior_points(labels: np.ndarray, affine: np.ndarray) -> dict[str, list[float]]:
    """For each organ, the voxel farthest from its edge (distance transform), in world mm."""
    out = {}
    for o, sl in zip(ORGANS, ndi.find_objects(labels, max_label=len(ORGANS))):
        if sl is None:
            continue
        m = np.pad(labels[sl] == o.index, 1)
        d = ndi.distance_transform_edt(m)[1:-1, 1:-1, 1:-1]
        k = np.unravel_index(int(np.argmax(d)), d.shape)
        vox = [s.start + int(kk) for s, kk in zip(sl, k)]
        mm = affine @ np.array([*vox, 1.0])
        out[o.key] = [round(float(v), 1) for v in mm[:3]]
    return out


def _save_labels(data: np.ndarray, like: nib.Nifti1Image, path: Path) -> None:
    img = nib.Nifti1Image(data.astype(np.uint8), like.affine)
    img.header.set_intent("label")
    img.header.set_xyzt_units("mm")
    nib.save(img, str(path))


def export_case(case_id: str, case_dir: str | Path, web_dir: str | Path, work_dir: str | Path,
                spacing_mm: float = 2.0) -> dict:
    case_dir, web_dir, work_dir = Path(case_dir), Path(web_dir), Path(work_dir)
    out = web_dir / case_id
    out.mkdir(parents=True, exist_ok=True)

    ref, ct = reference_labels(case_dir)
    ai_path = work_dir / "ai_raw" / f"{case_id}.nii.gz"
    ai = ai_labels(ai_path, ct)[0] if ai_path.exists() else None

    # 1) reorient to RAS so all cases open the same way
    ct_c = nib.as_closest_canonical(ct)
    ref_c = nib.as_closest_canonical(nib.Nifti1Image(ref, ct.affine))
    ai_c = nib.as_closest_canonical(nib.Nifti1Image(ai, ct.affine)) if ai is not None else None

    # 2) crop to the body (plus every labelled voxel, so no organ is ever cut)
    arr = np.asarray(ct_c.dataobj, dtype=np.float32)
    box = _bbox(arr > -500)
    lab_any = np.asarray(ref_c.dataobj) > 0
    if ai_c is not None:
        lab_any |= np.asarray(ai_c.dataobj) > 0
    if lab_any.any():
        box = _union(box, _bbox(lab_any, min_count=1, margin=4))
    del arr
    ct_crop = ct_c.slicer[box[0], box[1], box[2]]

    # 3) resample to a lighter grid (never finer than the original)
    zooms = [float(z) for z in ct_crop.header.get_zooms()[:3]]
    target = [max(spacing_mm, z) for z in zooms]
    ct_web = resample_to_output(ct_crop, voxel_sizes=target, order=1, mode="constant", cval=-1024)
    ct_data = np.clip(np.rint(np.asarray(ct_web.dataobj)), -1024, 3071).astype(np.int16)
    ct_img = nib.Nifti1Image(ct_data, ct_web.affine)
    ct_img.header["cal_min"], ct_img.header["cal_max"] = CT_WINDOW
    ct_img.header.set_xyzt_units("mm")
    nib.save(ct_img, str(out / "ct.nii.gz"))

    def to_web(lab_img):
        crop = lab_img.slicer[box[0], box[1], box[2]]
        return np.asarray(resample_from_to(crop, ct_img, order=0, mode="constant", cval=0).dataobj)

    ref_web = to_web(ref_c)
    _save_labels(ref_web, ct_img, out / "reference.nii.gz")
    files = {"ct": f"{case_id}/ct.nii.gz", "reference": f"{case_id}/reference.nii.gz"}
    ai_web = None
    if ai_c is not None:
        ai_web = to_web(ai_c)
        _save_labels(ai_web, ct_img, out / "ai.nii.gz")
        _save_labels(ai, ct, out / "ai_fullres.nii.gz")
        files["ai"] = f"{case_id}/ai.nii.gz"
        files["ai_fullres"] = f"{case_id}/ai_fullres.nii.gz"

    # one point deep inside each organ (world mm), so the viewer can jump there when a name is clicked.
    # (A plain centre of mass can fall outside curved organs such as the pancreas.)
    focus = {}
    for src in (ref_web, ai_web):
        if src is not None:
            for key, mm in _interior_points(src, ct_img.affine).items():
                focus.setdefault(key, mm)

    metrics_file = work_dir / "metrics" / f"{case_id}.json"
    metrics = json.loads(metrics_file.read_text()) if metrics_file.exists() else None
    if metrics:
        with open(out / "metrics.csv", "w", newline="") as f:
            w = csv.writer(f)
            w.writerow(["organ", "dice", "expert_ml", "ai_ml", "difference_pct"])
            for m in metrics["organs"].values():
                w.writerow([m["name"], m["dice"], m["vol_ref_ml"], m["vol_ai_ml"], m["vol_diff_pct"]])
        files["metrics_csv"] = f"{case_id}/metrics.csv"

    sizes = {k: round((web_dir / v).stat().st_size / 1e6, 2) for k, v in files.items()}
    return {
        "id": case_id,
        "files": files,
        "size_mb": sizes,
        "web_spacing_mm": [round(t, 2) for t in target],
        "focus_mm": focus,
        "metrics": metrics,
    }


def export_cases(case_ids: list[str], cases: dict[str, Path], web_dir: str | Path, work_dir: str | Path,
                 meta: dict[str, dict] | None = None, spacing_mm: float = 2.0, title_prefix: str = "Case",
                 extra: dict | None = None) -> Path:
    """Export several cases and write cases.json (the viewer's manifest)."""
    from .dataset import describe

    web_dir, work_dir = Path(web_dir), Path(work_dir)
    web_dir.mkdir(parents=True, exist_ok=True)
    entries = []
    for n, cid in enumerate(case_ids, 1):
        e = export_case(cid, cases[cid], web_dir, work_dir, spacing_mm)
        m = (meta or {}).get(cid, {})
        e["title"] = f"{title_prefix} {n}"
        e["description"] = describe(m)
        e["pathology"] = m.get("pathology", "")
        entries.append(e)
        big = [k for k, v in e["size_mb"].items() if v > 25]
        print(f"  {cid}: " + ", ".join(f"{k} {v} MB" for k, v in e["size_mb"].items())
              + ("   <-- larger than 25 MB, consider --web-spacing 2.5" if big else ""))

    manifest = {
        "app": "CT Anatomy Explorer",
        "version": "0.5",
        "generated": dt.datetime.now().strftime("%Y-%m-%d %H:%M"),
        "model": "VISTA-3D (MONAI bundle 'vista3d', label prompts for 12 organs)",
        "dataset": "TotalSegmentator small subset v2.0.1 (Wasserthal et al., Radiology: AI 2023), CC BY 4.0",
        "ct_window": list(CT_WINDOW),
        "organs": organ_table(),
        "compare": {str(k): {"name": v[0], "color": list(v[1])} for k, v in COMPARE.items()},
        "cases": entries,
    }
    if extra:
        manifest.update(extra)
    (web_dir / "cases.json").write_text(json.dumps(manifest, indent=1))
    print(f"Manifest written: {web_dir/'cases.json'} ({len(entries)} cases)")
    return web_dir / "cases.json"


def zip_web_data(web_dir: str | Path, zip_base: str | Path) -> Path:
    """Zip the web data folder so it can be downloaded in one click."""
    return Path(shutil.make_archive(str(zip_base), "zip", root_dir=str(Path(web_dir).parent),
                                    base_dir=Path(web_dir).name))


In [ ]:
%%writefile pipeline/run_all.py
"""One command for the whole pipeline (handy on a rented GPU or a lab PC).

    python -m pipeline.run_all --n-cases 10 --n-web 5

Steps (each one skips work that is already done, so you can re-run safely):
    download  -> curate -> infer -> evaluate -> export
"""
from __future__ import annotations

import argparse
import json
from pathlib import Path

from . import dataset, evaluate, export_web, run_vista3d


def default_paths() -> dict:
    """Kaggle-friendly defaults: big downloads in /tmp, results in /kaggle/working."""
    kaggle = Path("/kaggle/working").exists()
    work = Path("/kaggle/working/outputs") if kaggle else Path("outputs")
    return {
        "data_dir": Path("/tmp/totalseg") if kaggle else Path("data/totalseg"),
        "bundle_dir": Path("/tmp/bundles/vista3d") if kaggle else Path("data/bundles/vista3d"),
        "work_dir": work,
        "web_dir": work / "web_data" / "data",
    }


def main(argv=None) -> None:
    d = default_paths()
    p = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--data-dir", type=Path, default=d["data_dir"])
    p.add_argument("--bundle-dir", type=Path, default=d["bundle_dir"])
    p.add_argument("--work-dir", type=Path, default=d["work_dir"])
    p.add_argument("--web-dir", type=Path, default=None, help="default: <work-dir>/web_data/data")
    p.add_argument("--zip-path", type=Path, default=None, help="use an already downloaded dataset zip")
    p.add_argument("--n-cases", type=int, default=10, help="scans to segment (demo + spares)")
    p.add_argument("--n-web", type=int, default=5, help="scans to export to the viewer")
    p.add_argument("--cases", default=None, help="comma-separated case ids (skips automatic selection)")
    p.add_argument("--steps", default="download,curate,infer,evaluate,export")
    p.add_argument("--web-spacing", type=float, default=2.0, help="mm; larger = smaller files")
    p.add_argument("--allow-cpu", action="store_true", help="run VISTA-3D without a GPU (very slow)")
    a = p.parse_args(argv)
    steps = set(a.steps.split(","))
    web_dir = a.web_dir or a.work_dir / "web_data" / "data"
    a.work_dir.mkdir(parents=True, exist_ok=True)

    if "download" in steps:
        dataset.prepare(a.data_dir, a.zip_path)
    cases = dataset.find_cases(a.data_dir)
    sel_file = a.work_dir / "selected_cases.json"
    if a.cases:
        selected = [c.strip() for c in a.cases.split(",") if c.strip()]
    elif "curate" in steps or not sel_file.exists():
        selected = dataset.curate(a.data_dir, a.work_dir, a.n_cases)
    else:
        selected = json.loads(sel_file.read_text())

    if "infer" in steps:
        import torch
        if not torch.cuda.is_available() and not a.allow_cpu:
            raise SystemExit("No GPU found. Turn on a GPU accelerator (or pass --allow-cpu to wait for hours).")
        run_vista3d.ensure_bundle(a.bundle_dir)
        run_vista3d.run_cases(selected, cases, a.bundle_dir, a.work_dir)
    if "evaluate" in steps:
        evaluate.evaluate_cases(selected, cases, a.work_dir)
        for cid in selected:
            evaluate.qc_figure(cid, cases[cid], a.work_dir / "ai_raw" / f"{cid}.nii.gz", a.work_dir / "qc" / f"{cid}.png")
    if "export" in steps:
        done = [c for c in selected if (a.work_dir / "ai_raw" / f"{c}.nii.gz").exists()]
        export_web.export_cases(done[: a.n_web], cases, web_dir, a.work_dir, dataset.read_meta(a.data_dir),
                                a.web_spacing)
        z = export_web.zip_web_data(web_dir, a.work_dir / "web_data")
        print(f"Ready to download: {z}")


if __name__ == "__main__":
    main()


In [ ]:
import importlib, json, sys
from pathlib import Path
import pandas as pd
from IPython.display import Image, display
import pipeline, pipeline.organs, pipeline.dataset, pipeline.run_vista3d, pipeline.evaluate, pipeline.export_web, pipeline.run_all
for m in (pipeline.organs, pipeline.dataset, pipeline.run_vista3d, pipeline.evaluate, pipeline.export_web, pipeline.run_all):
    importlib.reload(m)   # picks up edits if you re-run the cells above
from pipeline import dataset, run_vista3d, evaluate, export_web
from pipeline.run_all import default_paths
P = default_paths()
for k, v in P.items():
    print(f'{k:10s} {v}')

## Step 1 · Download the dataset (3.2 GB, about 5–10 minutes)
TotalSegmentator small subset v2.0.1 from Zenodo (CC BY 4.0). Only the CT scans and the 12 organ masks are unpacked.

In [ ]:
dataset.prepare(P['data_dir'])
cases = dataset.find_cases(P['data_dir'])
print(len(cases), 'scans available')

## Step 2 · Choose the best scans
Keeps scans where all 12 organs are fully inside the image. The full ranking is saved as `outputs/curation.csv`.

In [ ]:
selected = dataset.curate(P['data_dir'], P['work_dir'], n_cases=N_CASES)
cur = pd.read_csv(P['work_dir'] / 'curation.csv')
display(cur.head(15)[['selected', 'case_id', 'organs_ok', 'description', 'shape', 'spacing_mm', 'length_mm']])

## Step 3 · Download the VISTA-3D model (about 870 MB)

In [ ]:
run_vista3d.ensure_bundle(P['bundle_dir'])

## Step 4 · Run VISTA-3D on the selected scans
Roughly 1–5 minutes per scan on a T4. Each scan runs separately, so one failure does not stop the rest. Time and peak GPU memory are recorded in `outputs/inference_log.csv` (write these in your Day 1 report).

In [ ]:
records = run_vista3d.run_cases(selected, cases, P['bundle_dir'], P['work_dir'])
display(pd.DataFrame(records)[['case_id', 'status', 'seconds', 'peak_gpu_mb']])

## Step 5 · Compare AI with the expert masks
Dice: 1.00 = identical. Lines starting with **!** are automatic warnings, e.g. a label mix-up or a left/right swap - investigate those before sharing any results.

In [ ]:
results = evaluate.evaluate_cases(selected, cases, P['work_dir'])
if results:
    df = pd.read_csv(P['work_dir'] / 'metrics_all.csv')
    display(df.pivot(index='organ', columns='case_id', values='dice').round(3))

## Step 6 · Visual check (your Day 1 proof screenshot)
Left to right: CT, expert masks, AI masks. Patient's right is on the left of each picture (radiology convention).

In [ ]:
for cid in selected:
    ai_path = P['work_dir'] / 'ai_raw' / f'{cid}.nii.gz'
    png = evaluate.qc_figure(cid, cases[cid], ai_path, P['work_dir'] / 'qc' / f'{cid}.png')
    display(Image(filename=str(png)))

## Step 7 · Export files for the web viewer
Creates `outputs/web_data.zip`. Unzip it into the repo's `docs/` folder (it contains a `data/` folder that replaces the demo data).

In [ ]:
done = [c for c in selected if (P['work_dir'] / 'ai_raw' / f'{c}.nii.gz').exists()]
export_web.export_cases(done[:N_WEB], cases, P['web_dir'], P['work_dir'], dataset.read_meta(P['data_dir']),
                        spacing_mm=WEB_SPACING_MM)
z = export_web.zip_web_data(P['web_dir'], P['work_dir'] / 'web_data')
print('Download this file:', z, f'({z.stat().st_size/1e6:.1f} MB)')

## Done – what to hand over
* `outputs/web_data.zip` → viewer team (unzip into `docs/`, so you get `docs/data/cases.json`).
* `outputs/qc/*.png` → post one in the group chat as the day's proof.
* `outputs/inference_log.csv`, `outputs/metrics_all.csv` → keep for the report.

**Credits:** TotalSegmentator dataset (Wasserthal et al., Radiology: AI 2023, CC BY 4.0); VISTA-3D (MONAI / NVIDIA).
Agreement scores are not a clinical validation.